# 🧪 LoRA Model Conversion & Interactive Gradio Chat

This notebook automates:
1. Downloading base models from Hugging Face Hub
2. Merging your fine-tuned LoRA adapters (from `Research Project/models_for_drive/`) into the base models
3. Converting the merged models to GGUF (`Q4_K_M`)
4. Saving `.gguf` files back to Google Drive (`Research Project/gguf/`)
5. Launching an interactive **Gradio Chat UI** with GPU acceleration (`-ngl 99`) and a model-picker dropdown for your demo!

In [5]:
# 1. Environment & GPU Sanity Check
import psutil, shutil, subprocess

ram = psutil.virtual_memory().total / 1e9
disk = shutil.disk_usage('/content')
print(f'System RAM : {ram:.1f} GB (Available: {psutil.virtual_memory().available/1e9:.1f} GB)')
print(f'Disk space : {disk.free/1e9:.1f} GB free of {disk.total/1e9:.1f} GB')

gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                     capture_output=True, text=True).stdout.strip()
print(f'GPU status : {gpu or "No GPU detected (Switch Runtime to T4 GPU)"}')

if 'T4' in gpu or 'GPU' in gpu:
    print('✅ GPU detected! Fast GPU-accelerated inference is enabled.')
else:
    print('⚠️ Please select Runtime > Change runtime type > T4 GPU for fast conversion & chat.')

System RAM : 13.6 GB (Available: 10.4 GB)
Disk space : 93.5 GB free of 115.7 GB


FileNotFoundError: [Errno 2] No such file or directory: 'nvidia-smi'

In [4]:
# 2. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

ValueError: Mountpoint must not already contain files

In [3]:
# 3. Model Configuration (Matched to your 'Research Project' Drive folder)
import os

# Target quantization format (Q4_K_M is the optimal sweet spot for speed & quality)
QUANT = 'Q4_K_M'

# Output directory in Research Project on Drive where .gguf files are saved permanently
OUT_DIR = '/content/drive/MyDrive/Research Project/gguf'
os.makedirs(OUT_DIR, exist_ok=True)

BASE_DRIVE_DIR = '/content/drive/MyDrive/Research Project/models_for_drive'

# Model Definitions:
# On Monday, simply add the 3 new models to this list!
MODELS = [
    {
        'name': 'Mistral-7B-Finetuned',
        'adapter_dir': f'{BASE_DRIVE_DIR}/mistral7b',
        'base_model': 'mistralai/Mistral-7B-Instruct-v0.3',
        'enabled': True
    },
    {
        'name': 'Granite-3.3-8B-Finetuned',
        'adapter_dir': f'{BASE_DRIVE_DIR}/granite33_8b',
        'base_model': 'ibm-granite/granite-3.3-8b-instruct',
        'enabled': True
    },
    {
        'name': 'Qwen3-14B-Finetuned',
        'adapter_dir': f'{BASE_DRIVE_DIR}/qwen3_14b',
        'base_model': 'Qwen/Qwen3-14B',
        'enabled': True
    },
    # --- Additional models can be added below on Monday: ---
    # {
    #     'name': 'Model-4-Finetuned',
    #     'adapter_dir': f'{BASE_DRIVE_DIR}/model4',
    #     'base_model': '...',
    #     'enabled': False
    # },
]

print(f'Target Quant : {QUANT}')
print(f'GGUF Output  : {OUT_DIR}\n')
print('Adapter Directory Check:')
for m in MODELS:
    if m['enabled']:
        found = os.path.isdir(m['adapter_dir'])
        status = '✅ Found on Drive' if found else '❌ Upload still in progress or check path'
        print(f"  • {m['name']:<25} : {status}")

Target Quant : Q4_K_M
GGUF Output  : /content/drive/MyDrive/Research Project/gguf

Adapter Directory Check:
  • Mistral-7B-Finetuned      : ✅ Found on Drive
  • Granite-3.3-8B-Finetuned  : ✅ Found on Drive
  • Qwen3-14B-Finetuned       : ✅ Found on Drive


In [4]:
%%bash
# 4. Toolchain Setup (llama.cpp with CUDA support + Python libraries)
set -e
mkdir -p /content/llama-bin

if [ ! -x /content/llama-bin/llama-cli ]; then
  echo '>>> Downloading pre-compiled llama.cpp linux-x64 binaries...'
  URL=$(curl -sL 'https://api.github.com/repos/ggml-org/llama.cpp/releases?per_page=15' | python3 -c "
import json, re, sys
pat = re.compile(r'llama-b\\d+-bin-ubuntu-x64\\.tar\\.gz\$')
for rel in json.load(sys.stdin):
    for a in rel['assets']:
        if pat.search(a['name']):
            print(a['browser_download_url']); raise SystemExit
")
  if [ -n "$URL" ]; then
    wget -q "$URL" -O /content/llama-bin.tar.gz
    mkdir -p /content/llama-extract
    tar -xzf /content/llama-bin.tar.gz -C /content/llama-extract
    find /content/llama-extract -type f -exec cp {} /content/llama-bin/ \;
    chmod +x /content/llama-bin/llama-* /content/llama-bin/ggml-* 2>/dev/null || true
  fi
fi

export LD_LIBRARY_PATH=/content/llama-bin:${LD_LIBRARY_PATH:-}

echo '>>> Ensuring llama.cpp source repo is cloned for convert_hf_to_gguf.py...'
[ -f /content/llama.cpp/convert_hf_to_gguf.py ] || git clone --depth 1 https://github.com/ggml-org/llama.cpp /content/llama.cpp

echo '>>> Installing required Python dependencies...'
pip install -q --upgrade pip
pip install -q safetensors sentencepiece protobuf numpy peft accelerate transformers gradio
pip install -q -e /content/llama.cpp/gguf-py

python3 -c 'import torch, transformers, peft, gguf, gradio; print("Environment ready! CUDA:", torch.cuda.is_available())'

>>> Downloading pre-compiled llama.cpp linux-x64 binaries...
>>> Ensuring llama.cpp source repo is cloned for convert_hf_to_gguf.py...
>>> Installing required Python dependencies...
Environment ready! CUDA: False


Cloning into '/content/llama.cpp'...
I0000 00:00:1793466185.029607    9698 pjrt_api.cc:119] GetPjrtApi was found for TPU at /usr/local/lib/python3.13/dist-packages/libtpu/libtpu.so
I0000 00:00:1793466185.912538    9698 device_rt.cc:166] PjRt runtime initialization deferred for tpu
Successfully renamed PrivateUse1 backend to 'tpu'. Device: device(type='tpu')
Registered Python module for 'tpu'.
Device type: tpu, Device index: default
Initializing TPU distributed runtime
I0000 00:00:1793466185.915531    9698 libtpu_version.cc:63] Setting libtpu version to: 0.0.50


In [2]:
# 5. Automated LoRA Merge & Batch GGUF Quantization (100% Autonomous & Streamed - Zero OOM)
import os, gc, time, shutil, subprocess, json, torch

# 1. Ensure Drive is mounted
if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive')

# 2. Toolchain Auto-Setup (Clones llama.cpp & installs llama-bin if missing)
if not os.path.exists('/content/llama.cpp/convert_hf_to_gguf.py'):
    print('>>> [Setup] Cloning llama.cpp converter repo...')
    subprocess.run('git clone --depth 1 https://github.com/ggml-org/llama.cpp /content/llama.cpp', shell=True)
    subprocess.run('pip install -q -e /content/llama.cpp/gguf-py', shell=True)

if not os.path.exists('/content/llama-bin/llama-quantize'):
    print('>>> [Setup] Downloading pre-compiled llama.cpp tools...')
    cmd = '''
    mkdir -p /content/llama-bin
    URL=$(curl -sL 'https://api.github.com/repos/ggml-org/llama.cpp/releases?per_page=15' | python3 -c "
import json, re, sys
pat = re.compile(r'llama-b\\d+-bin-ubuntu-x64\\.tar\\.gz$')
for rel in json.load(sys.stdin):
    for a in rel['assets']:
        if pat.search(a['name']):
            print(a['browser_download_url']); raise SystemExit
")
    if [ -n "$URL" ]; then
        wget -q "$URL" -O /content/llama-bin.tar.gz
        mkdir -p /content/llama-extract
        tar -xzf /content/llama-bin.tar.gz -C /content/llama-extract
        find /content/llama-extract -type f -exec cp {} /content/llama-bin/ \;
        chmod +x /content/llama-bin/llama-* /content/llama-bin/ggml-* 2>/dev/null || true
    fi
    '''
    subprocess.run(cmd, shell=True)

subprocess.run('cp -a /content/llama-bin/*.so* /usr/lib/ 2>/dev/null; ldconfig', shell=True)

# 3. Model Configuration
QUANT = 'Q4_K_M'
OUT_DIR = '/content/drive/MyDrive/Research Project/gguf'
os.makedirs(OUT_DIR, exist_ok=True)

BASE_DRIVE_DIR = '/content/drive/MyDrive/Research Project/models_for_drive'
MODELS = [
    {
        'name': 'Mistral-7B-Finetuned',
        'adapter_dir': f'{BASE_DRIVE_DIR}/mistral7b',
        'base_model': 'mistralai/Mistral-7B-Instruct-v0.3',
        'enabled': True
    },
    {
        'name': 'Granite-3.3-8B-Finetuned',
        'adapter_dir': f'{BASE_DRIVE_DIR}/granite33_8b',
        'base_model': 'ibm-granite/granite-3.3-8b-instruct',
        'enabled': True
    },
    {
        'name': 'Qwen3-14B-Finetuned',
        'adapter_dir': f'{BASE_DRIVE_DIR}/qwen3_14b',
        'base_model': 'Qwen/Qwen3-14B',
        'enabled': True
    },
]

from safetensors.torch import load_file, save_file
from huggingface_hub import snapshot_download

def merge_lora_turbo(base_repo, adapter_dir, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    print(f'   -> Locating base model snapshot for {base_repo}...')
    base_dir = snapshot_download(base_repo)

    with open(os.path.join(adapter_dir, 'adapter_config.json')) as f:
        cfg = json.load(f)
    r = cfg.get('r', 64)
    alpha = cfg.get('lora_alpha', 128)
    scale = float(alpha) / float(r)
    print(f'   -> LoRA configuration: rank={r}, alpha={alpha}, scaling factor={scale:.2f}')

    adapter_path = os.path.join(adapter_dir, 'adapter_model.safetensors')
    adapter_weights = load_file(adapter_path, device='cpu')
    print(f'   -> Loaded {len(adapter_weights)} adapter tensors ({os.path.getsize(adapter_path)/1e6:.1f} MB)')

    index_file = os.path.join(base_dir, 'model.safetensors.index.json')
    if os.path.exists(index_file):
        with open(index_file) as f:
            idx = json.load(f)
        shard_names = sorted(list(set(idx['weight_map'].values())))
    else:
        shard_names = sorted([f for f in os.listdir(base_dir) if f.endswith('.safetensors')])

    dev = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f'   -> Turbo Merging {len(shard_names)} weight shard(s) on {dev.upper()} (~15s/shard)...')

    for s_idx, s_name in enumerate(shard_names, 1):
        s_path = os.path.join(base_dir, s_name)
        out_path = os.path.join(output_dir, s_name)
        t_shard = time.time()
        print(f'      [{s_idx}/{len(shard_names)}] Processing {s_name} ({os.path.getsize(s_path)/1e9:.2f} GB)...', end='', flush=True)

        shard_tensors = load_file(s_path, device='cpu')
        applied = 0

        for k in list(shard_tensors.keys()):
            prefix = k[:-len('.weight')] if k.endswith('.weight') else k
            lora_a_key = f'base_model.model.{prefix}.lora_A.weight'
            lora_b_key = f'base_model.model.{prefix}.lora_B.weight'
            if lora_a_key not in adapter_weights:
                lora_a_key = f'{prefix}.lora_A.weight'
                lora_b_key = f'{prefix}.lora_B.weight'

            if lora_a_key in adapter_weights and lora_b_key in adapter_weights:
                a = adapter_weights[lora_a_key].to(dev, dtype=torch.float32)
                b = adapter_weights[lora_b_key].to(dev, dtype=torch.float32)
                v = shard_tensors[k].to(dev, dtype=torch.float32)
                
                shard_tensors[k] = (v + (b @ a) * scale).to('cpu', dtype=shard_tensors[k].dtype)
                applied += 1

        save_file(shard_tensors, out_path, metadata={'format': 'pt'})
        del shard_tensors
        if dev == 'cuda': torch.cuda.empty_cache()
        gc.collect()
        print(f' done in {time.time()-t_shard:.1f}s ({applied} layers fused)')

    del adapter_weights
    gc.collect()

    print('   -> Copying configs and tokenizer definitions...')
    for item in os.listdir(base_dir):
        if not item.endswith('.safetensors') and not item.endswith('.bin') and not item.startswith('.'):
            s = os.path.join(base_dir, item)
            d = os.path.join(output_dir, item)
            if os.path.isfile(s) and not os.path.exists(d):
                shutil.copy2(s, d)

    for item in os.listdir(adapter_dir):
        if (item.endswith('.json') or item.endswith('.jinja') or item.endswith('.model')) and item != 'adapter_config.json':
            shutil.copy2(os.path.join(adapter_dir, item), os.path.join(output_dir, item))

converted_models = {}

for idx, m in enumerate(MODELS, 1):
    if not m.get('enabled', False):
        continue

    m_name = m['name']
    adapter_dir = m['adapter_dir']
    base_repo = m['base_model']
    final_gguf = f"{OUT_DIR}/{m_name}-{QUANT}.gguf"

    print(f'\n{"="*65}')
    print(f'[{idx}/{len(MODELS)}] Model: {m_name}')
    print(f'{"="*65}')

    if os.path.exists(final_gguf) and os.path.getsize(final_gguf) > 1e8:
        print(f'✅ Already converted: {final_gguf} ({os.path.getsize(final_gguf)/1e9:.2f} GB)')
        converted_models[m_name] = final_gguf
        continue

    if not os.path.isdir(adapter_dir):
        print(f'❌ Skipped: Adapter folder not found at {adapter_dir}')
        continue

    t_start = time.time()
    merged_dir = f'/content/{m_name}-merged'
    intermediate_gguf = f'/content/{m_name}-f16.gguf'
    staging_gguf = f'/content/{m_name}-{QUANT}.gguf'

    # Step A: Turbo GPU-streamed merge
    print(f'1. Merging {base_repo} with LoRA adapter...')
    merge_lora_turbo(base_repo, adapter_dir, merged_dir)

    # Step B: Convert HF merged weights to intermediate GGUF (f16)
    print('2. Converting merged weights to intermediate GGUF (f16)...')
    cmd_convert = [
        'python3', '/content/llama.cpp/convert_hf_to_gguf.py', merged_dir,
        '--outfile', intermediate_gguf,
        '--outtype', 'f16'
    ]
    res_conv = subprocess.run(cmd_convert, capture_output=True, text=True)
    if res_conv.returncode != 0:
        print('❌ Conversion to intermediate GGUF failed:', res_conv.stderr[-500:])
        shutil.rmtree(merged_dir, ignore_errors=True)
        continue

    # Free merged directory space
    shutil.rmtree(merged_dir, ignore_errors=True)

    # Step C: Quantize to Q4_K_M
    print(f'3. Quantizing to {QUANT} via llama-quantize...')
    cmd_quant = ['/content/llama-bin/llama-quantize', intermediate_gguf, staging_gguf, QUANT]
    res_quant = subprocess.run(cmd_quant, capture_output=True, text=True)
    if res_quant.returncode != 0:
        print('❌ Quantization failed:', res_quant.stderr[-500:])
        if os.path.exists(intermediate_gguf): os.remove(intermediate_gguf)
        continue

    if os.path.exists(intermediate_gguf):
        os.remove(intermediate_gguf)

    # Step D: Save final GGUF to Google Drive
    print(f'4. Saving {QUANT} GGUF to Google Drive: {final_gguf}...')
    shutil.copy2(staging_gguf, final_gguf)
    if os.path.exists(staging_gguf):
        os.remove(staging_gguf)

    elapsed = (time.time() - t_start) / 60
    size_gb = os.path.getsize(final_gguf) / 1e9
    print(f'✅ Finished in {elapsed:.1f} min! File size: {size_gb:.2f} GB')
    converted_models[m_name] = final_gguf

print(f'\n{"="*65}')
print(f'Pipeline Complete! {len(converted_models)} model(s) ready in Google Drive:')
for name, path in converted_models.items():
    print(f'  • {name:<25}: {os.path.getsize(path)/1e9:.2f} GB')



NameError: name 'MODELS' is not defined

In [1]:
%%bash
# 1. Register all llama libraries with Linux system so they are always found
cp -a /content/llama-extract/*/* /usr/lib/ 2>/dev/null || true
cp -a /content/llama-bin/*.so* /usr/lib/ 2>/dev/null || true
ldconfig

# 2. Quantize the already-converted Mistral file directly to your Google Drive!
mkdir -p "/content/drive/MyDrive/Research Project/gguf"
/content/llama-bin/llama-quantize /content/Mistral-7B-Finetuned-f16.gguf "/content/drive/MyDrive/Research Project/gguf/Mistral-7B-Finetuned-Q4_K_M.gguf" Q4_K_M

/sbin/ldconfig.real: /usr/local/lib/libtbbbind_2_0.so.3 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbbmalloc.so.2 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libur_adapter_level_zero.so.0 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbbbind_2_5.so.3 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtcm.so.1 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbbmalloc_proxy.so.2 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libhwloc.so.15 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libur_adapter_level_zero_v2.so.0 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libur_adapter_opencl.so.0 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libumf.so.1 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbbbind.so.3 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libur_loader.so.0 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/

CalledProcessError: Command 'b'# 1. Register all llama libraries with Linux system so they are always found\ncp -a /content/llama-extract/*/* /usr/lib/ 2>/dev/null || true\ncp -a /content/llama-bin/*.so* /usr/lib/ 2>/dev/null || true\nldconfig\n\n# 2. Quantize the already-converted Mistral file directly to your Google Drive!\nmkdir -p "/content/drive/MyDrive/Research Project/gguf"\n/content/llama-bin/llama-quantize /content/Mistral-7B-Finetuned-f16.gguf "/content/drive/MyDrive/Research Project/gguf/Mistral-7B-Finetuned-Q4_K_M.gguf" Q4_K_M\n'' returned non-zero exit status 127.

In [8]:
%%bash
# 1. Remove conflicting pre-installed libraries
pip uninstall -y torchao

# 2. Install fully compatible versions of all required packages in one shot
pip install -q "numpy<2.3" -U scipy
pip install -q --upgrade pip
pip install -q safetensors sentencepiece protobuf peft accelerate transformers gradio

# 3. Clone and install llama.cpp python tools
[ -d /content/llama.cpp ] || git clone --depth 1 https://github.com/ggml-org/llama.cpp /content/llama.cpp
pip install -q -e /content/llama.cpp/gguf-py

# 4. Download pre-compiled llama.cpp GPU/CPU binaries
mkdir -p /content/llama-bin
if [ ! -x /content/llama-bin/llama-cli ]; then
  URL=$(curl -sL 'https://api.github.com/repos/ggml-org/llama.cpp/releases?per_page=15' | python3 -c "
import json, re, sys
pat = re.compile(r'llama-b\\d+-bin-ubuntu-x64\\.tar\\.gz$')
for rel in json.load(sys.stdin):
    for a in rel['assets']:
        if pat.search(a['name']):
            print(a['browser_download_url']); raise SystemExit
")
  if [ -n "$URL" ]; then
    wget -q "$URL" -O /content/llama-bin.tar.gz
    mkdir -p /content/llama-extract
    tar -xzf /content/llama-bin.tar.gz -C /content/llama-extract
    find /content/llama-extract -type f -exec cp {} /content/llama-bin/ \;
    chmod +x /content/llama-bin/llama-* /content/llama-bin/ggml-* 2>/dev/null || true
  fi
fi

# 5. Verify entire environment is clean
python3 -c 'import torch, transformers, peft, gguf, gradio; print("🔥 ALL DEPENDENCIES READY! CUDA:", torch.cuda.is_available())'


🔥 ALL DEPENDENCIES READY! CUDA: False


I0000 00:00:1793466314.022671   10495 pjrt_api.cc:119] GetPjrtApi was found for TPU at /usr/local/lib/python3.13/dist-packages/libtpu/libtpu.so
I0000 00:00:1793466314.899231   10495 device_rt.cc:166] PjRt runtime initialization deferred for tpu
Successfully renamed PrivateUse1 backend to 'tpu'. Device: device(type='tpu')
Registered Python module for 'tpu'.
Device type: tpu, Device index: default
Initializing TPU distributed runtime
I0000 00:00:1793466314.902302   10495 libtpu_version.cc:63] Setting libtpu version to: 0.0.50


In [7]:
!pip install "numpy<2.3" -U scipy


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 46.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 114.9 MB/s  0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.5.4
    Uninstalling numpy-2.5.4:
      Successfully uninstalled numpy-2.5.4
  Attempting uninstall: scipy
    Found existing installation: scipy 1.16.3
    Uninstalling scipy-1.16.3:
      Successfully uninstalled scipy-1.16.3
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [scipy]


In [8]:
!cp "/content/drive/MyDrive/Research Project/gguf/Mistral-7B-Finetuned-Q4_K_M.gguf" /content/

In [14]:
# 💬 Complete Live Chat Interface with Streaming (Powered by llama-server)
import os, glob, subprocess, time, json, requests, socket

# 1. Mount Google Drive if available
if not os.path.exists('/content/drive/MyDrive'):
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except Exception as e:
        print(f"Drive notice: {e}")

# 2. Ensure Gradio is installed
try:
    import gradio as gr
except ImportError:
    os.system('pip install -q gradio')
    import gradio as gr

# 3. Setup llama.cpp server binaries (~15s on fresh runtime)
SERVER = '/content/llama-bin/llama-server'
if not os.path.exists(SERVER):
    print("Setting up llama.cpp server engine (~15s)...")
    os.makedirs('/content/llama-bin', exist_ok=True)
    os.system('wget -q https://github.com/ggml-org/llama.cpp/releases/download/b12020/llama-b12020-bin-ubuntu-x64.tar.gz -O /content/llama-bin.tar.gz')
    os.system('tar -xzf /content/llama-bin.tar.gz -C /content/llama-bin --strip-components=1')
    os.system('chmod +x /content/llama-bin/llama-*')
    os.system('cp -a /content/llama-bin/*.so* /usr/lib/ 2>/dev/null || true')
    os.system('ldconfig')

os.environ['LD_LIBRARY_PATH'] = '/content/llama-bin:' + os.environ.get('LD_LIBRARY_PATH', '')

# 4. Search for GGUF models in /content (fast SSD) and Google Drive
search_paths = [
    '/content',
    '/content/drive/MyDrive/Research Project/gguf',
    '/content/drive/MyDrive/Research project/gguf',
    '/content/drive/MyDrive/gguf'
]

available_ggufs = {}
for path in search_paths:
    if os.path.exists(path):
        for f in sorted(glob.glob(f'{path}/*.gguf')):
            label = os.path.basename(f).replace('.gguf', '')
            if label not in available_ggufs:
                available_ggufs[label] = f

if not available_ggufs:
    raise FileNotFoundError("❌ No .gguf files found in /content or Google Drive!")

print(f"\n✅ Ready models found:")
for k, v in available_ggufs.items():
    print(f"  • {k} ({os.path.getsize(v)/1e9:.2f} GB) at {v}")

# 5. Dynamically assign a guaranteed free port (avoids Colab 8080 collision!)
def get_free_port():
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.bind(('127.0.0.1', 0))
        return s.getsockname()[1]

SERVER_PORT = get_free_port()
print(f"📡 Dedicated llama-server port: {SERVER_PORT}")

# 6. Background llama-server manager (Keeps model in memory & streams responses)
current_server_proc = None
current_loaded_model = None

def ensure_server(model_label):
    global current_server_proc, current_loaded_model
    gguf_path = available_ggufs[model_label]

    # Check if this model is already active and healthy
    if current_loaded_model == model_label and current_server_proc and current_server_proc.poll() is None:
        try:
            r = requests.get(f'http://127.0.0.1:{SERVER_PORT}/health', timeout=1)
            if r.status_code == 200:
                return True, ""
        except Exception:
            pass

    # Clean up any existing server instance
    if current_server_proc:
        try:
            current_server_proc.terminate()
            current_server_proc.wait(timeout=3)
        except Exception:
            try: current_server_proc.kill()
            except Exception: pass
        current_server_proc = None

    os.system(f'fuser -k {SERVER_PORT}/tcp 2>/dev/null || true')
    time.sleep(1)

    print(f"⚡ Starting llama-server for '{model_label}' on port {SERVER_PORT}...")
    log_file = open('/tmp/llama_server.log', 'w')

    cmd = [
        SERVER,
        '-m', gguf_path,
        '--host', '127.0.0.1',
        '--port', str(SERVER_PORT),
        '-c', '2048',       # 2k context window (lightweight on RAM & CPU)
        '-t', '2'           # 2 threads matching Colab vCPU
    ]
    current_server_proc = subprocess.Popen(cmd, stdout=log_file, stderr=log_file)
    current_loaded_model = model_label

    # Wait for server readiness (up to 45 seconds for CPU mmap)
    for i in range(45):
        if current_server_proc.poll() is not None:
            log_file.close()
            with open('/tmp/llama_server.log', 'r') as f:
                err_log = f.read()
            print(f"❌ Server exited prematurely:\n{err_log}")
            return False, err_log
        try:
            r = requests.get(f'http://127.0.0.1:{SERVER_PORT}/health', timeout=1)
            if r.status_code == 200:
                print("✅ Model loaded into RAM and llama-server is ready!")
                return True, ""
        except Exception:
            pass
        time.sleep(1)

    return False, "Server took longer than 45s to load weights."

# Pre-load the first model into memory right now before launching UI
model_options = list(available_ggufs.keys())
print(f"\nInitializing model '{model_options[0]}' into memory...")
ok, err = ensure_server(model_options[0])
if not ok:
    print(f"⚠️ Warning during initial pre-load: {err}")

# 7. Real-time Streaming Chat Handler (Never freezes or times out!)
def chat_handler(user_msg, history, selected_model, max_tokens, temp, top_p):
    ready, err_msg = ensure_server(selected_model)
    if not ready:
        yield f"Error loading '{selected_model}':\n{err_msg[-400:] if err_msg else 'Timed out waiting for server.'}"
        return

    # Build standard OpenAI messages payload
    messages = [{"role": "system", "content": "You are a helpful, knowledgeable AI assistant."}]
    for turn in history:
        if isinstance(turn, (list, tuple)):
            u, b = turn[0], turn[1]
            if u: messages.append({"role": "user", "content": u})
            if b: messages.append({"role": "assistant", "content": b})
        elif isinstance(turn, dict):
            messages.append({"role": turn.get("role", "user"), "content": turn.get("content", "")})
    messages.append({"role": "user", "content": user_msg})

    payload = {
        "messages": messages,
        "max_tokens": int(max_tokens),
        "temperature": float(temp),
        "top_p": float(top_p),
        "stream": True
    }

    try:
        response = requests.post(f"http://127.0.0.1:{SERVER_PORT}/v1/chat/completions", json=payload, stream=True, timeout=120)
        accumulated = ""
        for line in response.iter_lines():
            if line:
                line_str = line.decode('utf-8')
                if line_str.startswith('data: '):
                    data_str = line_str[6:].strip()
                    if data_str == '[DONE]':
                        break
                    try:
                        data = json.loads(data_str)
                        delta = data['choices'][0]['delta'].get('content', '')
                        if delta:
                            accumulated += delta
                            yield accumulated
                    except Exception:
                        pass
        if not accumulated:
            yield "No response generated by model."
    except Exception as e:
        yield f"Inference error: {e}"

# 8. Gradio UI
with gr.Blocks(title="Model Evaluation & Demo", theme=gr.themes.Soft(primary_hue="indigo")) as demo:
    gr.Markdown("# 🚀 Fine-Tuned LLM Live Demonstration")
    gr.Markdown("Real-time streaming inference powered by `llama.cpp` server.")

    with gr.Row():
        with gr.Column(scale=1):
            model_selector = gr.Dropdown(choices=model_options, value=model_options[0], label="🤖 Select Model", interactive=True)
            max_tokens_slider = gr.Slider(32, 512, value=128, step=16, label="Max Tokens")
            temp_slider = gr.Slider(0.0, 1.5, value=0.7, step=0.05, label="Temperature")
            top_p_slider = gr.Slider(0.1, 1.0, value=0.9, step=0.05, label="Top-p")
            gr.Markdown(
                "### 💡 Demo Tips\n"
                "- **Streaming Active**: Responses start appearing immediately word-by-word!\n"
                "- **Persistent Memory**: The model stays loaded in RAM, making all responses fast."
            )
        with gr.Column(scale=3):
            chat = gr.ChatInterface(
                fn=chat_handler,
                additional_inputs=[model_selector, max_tokens_slider, temp_slider, top_p_slider],
                chatbot=gr.Chatbot(height=520)
            )

demo.launch(share=True, debug=False)


✅ Ready models found:
  • Mistral-7B-Finetuned-Q4_K_M (4.37 GB) at /content/Mistral-7B-Finetuned-Q4_K_M.gguf
📡 Dedicated llama-server port: 55923

Initializing model 'Mistral-7B-Finetuned-Q4_K_M' into memory...
⚡ Starting llama-server for 'Mistral-7B-Finetuned-Q4_K_M' on port 55923...
✅ Model loaded into RAM and llama-server is ready!


/tmp/ipykernel_5114/2440365119.py:184: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(title="Model Evaluation & Demo", theme=gr.themes.Soft(primary_hue="indigo")) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://289db407a80aea8cca.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
